# بررسی خروجی‌های pipeline رادیومیکس

این notebook را **بعد از اجرای pipeline** باز کنید (روی دادهٔ مصنوعی: `python tools/run_demo.py`).
هر cell را با `Shift+Enter` اجرا کنید و خروجی را با توضیح بالای آن مقایسه کنید.

هدف: ببینید هر مرحله دقیقاً چه چیزی تولید می‌کند و یاد بگیرید چطور خروجی‌ها را کنترل کیفیت کنید.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, RESULTS = ROOT / "data", ROOT / "results"
MASK = "reader1"      # main analysis; try "auto" or "reader2"
BW = 0.5              # bin width used in step 4

if not (DATA / "nifti").exists():
    raise SystemExit("data/nifti not found: this notebook is for PHASE 1 (images). Run 'python tools/run_demo.py' first "
                     "(needs PyRadiomics). For the phase-2 results open notebooks/phase2_results.ipynb instead.")
patients = sorted(p.name for p in (DATA / "nifti").iterdir() if p.is_dir())
PID = patients[0]     # change to look at another patient
print(len(patients), "patients:", patients)

## ۱. تصویر SUV و ماسک تومور (قدم ۱ و ۲)

**MIP** (Maximum Intensity Projection) همان نمایی است که پزشک طب هسته‌ای اول نگاه می‌کند.
خط نارنجی مرز ماسک تومور است. بررسی کنید:
- جذب فیزیولوژیک (کلیه، مثانه، غدد بزاقی) **داخل** ماسک نباشد
- ضایعه‌ای جا نیفتاده باشد

In [ ]:
pet = sitk.ReadImage(str(DATA / "nifti" / PID / "PET_SUV.nii.gz"))
mask = sitk.ReadImage(str(DATA / "masks" / PID / f"tumor_mask_{MASK}.nii.gz"))
pet_a, mask_a = sitk.GetArrayFromImage(pet), sitk.GetArrayFromImage(mask) > 0   # (z, y, x)
sx, sy, sz = pet.GetSpacing()

fig, axes = plt.subplots(1, 2, figsize=(9, 4), layout="constrained")
for ax, axis, aspect, title in [(axes[0], 1, sz / sx, "Coronal MIP"), (axes[1], 2, sz / sy, "Sagittal MIP")]:
    ax.imshow(pet_a.max(axis=axis), cmap="gray_r", vmin=0, vmax=10, origin="lower", aspect=aspect)
    ax.contour(mask_a.max(axis=axis), levels=[0.5], colors="#eb6834", linewidths=1.2, origin="lower")
    ax.set_title(f"{PID} - {title}", fontsize=11, color="#0b0b0b")
    ax.axis("off")
fig.supxlabel("Grey: SUV 0-10   Orange outline: tumour mask", color="#52514e", fontsize=10)
plt.show()
print(f"SUVmax in image: {pet_a.max():.1f}   voxel: {sx:.1f} x {sy:.1f} x {sz:.1f} mm")

## ۲. پارامترهای مرسوم PET (قدم ۳)

هر سطر یک ضایعه است؛ ضایعهٔ ۱ داغ‌ترین است. `suvmean` و `psma_tv_ml` در سطح بیمار متغیرهای **مدل مرجع** پروپوزال‌اند.

In [ ]:
lesions = pd.read_csv(RESULTS / MASK / "conventional_lesion.csv")
display(lesions[lesions.patient_id == PID].round(2))

conv = pd.read_csv(RESULTS / MASK / "conventional_patient.csv").set_index("patient_id")
display(conv.round(2))

## ۳. ویژگی‌های رادیومیکس در سطح بیمار (قدم ۴ و ۵)

نام هر ستون از سه بخش ساخته شده: `<روش تجمیع>_original_<کلاس>_<ویژگی>`
- `wb_` = union کل‌بدن، `vwm_` = میانگین وزنی حجمی، `sd_` / `range_` = ناهمگونی بین ضایعات
- مثال: `vwm_original_glszm_ZoneEntropy` = میانگین وزنی ZoneEntropy ضایعات

In [ ]:
feat = pd.read_csv(RESULTS / MASK / f"patient_features_pet_bw{BW:g}.csv").set_index("patient_id")
groups = feat.columns.str.extract(r"^(wb|vwm|sd|range)_original_([a-z]+)_")[[0, 1]].value_counts().unstack(fill_value=0)
print("Features per aggregation (rows) and class (columns):")
display(groups)
print("Patients x columns:", feat.shape)
feat[["suvmean", "psma_tv_ml", "wb_original_firstorder_Entropy", "sd_original_firstorder_Mean"]].round(3)

## ۴. تکرارپذیری بین دو Reader (قدم ۶)

فقط ویژگی‌های سمت راست خط چین (ICC ≥ 0.75) وارد مدل‌سازی می‌شوند.
روی دادهٔ مصنوعی ICCها پایین است چون تغییر ماسک Reader 2 عمداً بزرگ شبیه‌سازی شده؛ روی دادهٔ واقعی انتظار دارید first-order بالاتر از texture باشد.

In [ ]:
icc = pd.read_csv(RESULTS / f"icc_pet_bw{BW:g}.csv")
rad = icc[icc.feature.str.match(r"^(wb|vwm|sd|range)_")].dropna(subset=["icc"])

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(rad.icc.clip(-0.2, 1), bins=np.linspace(-0.2, 1, 25), color="#2a78d6", edgecolor="white", linewidth=1)
ax.axvline(0.75, color="#52514e", linestyle="--", linewidth=1)
ax.text(0.76, ax.get_ylim()[1] * 0.9, "ICC 0.75 cut-off", color="#52514e", fontsize=9)
ax.set_xlabel("ICC(2,1) between reader 1 and reader 2", color="#52514e")
ax.set_ylabel("Number of features", color="#52514e")
ax.set_title(f"Inter-reader reproducibility of {len(rad)} radiomic features", fontsize=11, loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e4df", linewidth=0.8); ax.set_axisbelow(True)
plt.show()

print(f"{rad.reproducible.sum()} of {len(rad)} features kept")
display(rad.sort_values("icc", ascending=False).head(10).round(3))

## ۵. endpointها (قدم ۷)

این جدول **جدا** از ویژگی‌های تصویری نگه داشته می‌شود و فقط در فاز مدل‌سازی (بعد از ثبت OSF) به آن وصل می‌شود.
`recur_event`: 0 = سانسور، 1 = عود، 2 = مرگ بدون عود (competing)؛ فقط برای پاسخ‌دهنده‌ها مقدار دارد.

In [ ]:
out = pd.read_csv(RESULTS / "outcomes.csv")
display(out.round(1))
print(out.temporal_set.value_counts())

## تمرین‌ها (برای یادگیری)
1. `PID` را عوض کنید و MIP بیمار دیگری را ببینید. کدام بیمار بیشترین PSMA-TV را دارد؟
2. `MASK = "auto"` بگذارید؛ تعداد ضایعات با `reader1` چه فرقی دارد؟
3. در ترمینال `python -m pipeline.step4_extract_radiomics --bin-width 0.25` و سپس step5 با همان bin width را اجرا کنید، `BW = 0.25` بگذارید و ببینید کدام ویژگی‌ها بیشترین تغییر را دارند.
4. همبستگی `suvmean` و `wb_original_firstorder_Mean` را حساب کنید (`feat[[...]].corr()`). چرا تقریباً ۱ است؟